# LAnoBERT trên Kaggle - chạy từng bước

Notebook này dành cho Kaggle GPU. Hãy upload repository `LAnoBERT` thành Kaggle Dataset, attach dataset vào notebook, rồi chạy từng cell theo thứ tự.

Notebook dùng `/kaggle/working/LAnoBERT` để tránh sửa dữ liệu trong `/kaggle/input`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

PROJECT_INPUT = Path('/kaggle/input/lanobert')  # đổi tên nếu Dataset khác
PROJECT = Path('/kaggle/working/LAnoBERT')
CONFIG = 'configs/bgl.yaml'
DOWNLOAD_BGL = False  # True nếu bật Internet trên Kaggle và chưa upload BGL.log
EPOCHS = 1             # đổi thành 10 cho thực nghiệm đầy đủ
MAX_EVAL_SAMPLES = 10000  # None để đánh giá toàn bộ test set

assert PROJECT_INPUT.exists(), f'Không tìm thấy {PROJECT_INPUT}; hãy sửa PROJECT_INPUT.'
+    print('Input:', PROJECT_INPUT)
+    print('Working:', PROJECT)

In [ ]:
import torch
+    print('Python:', sys.version)
+    print('PyTorch:', torch.__version__)
+    print('CUDA available:', torch.cuda.is_available())
+    if torch.cuda.is_available():
+        print('GPU:', torch.cuda.get_device_name(0))
+    else:
+        print('Cảnh báo: chưa bật GPU accelerator trong Kaggle.')

In [ ]:
if PROJECT.exists():
+    shutil.rmtree(PROJECT)
+
+    def ignore(path, names):
+        ignored = {'.git', '__pycache__', '.pytest_cache', 'outputs', 'data'}
+        return [name for name in names if name in ignored]
+
+    source = PROJECT_INPUT
+    if not (source / 'lanobert').is_dir():
+        candidates = [p.parent for p in source.rglob('lanobert')
+                      if p.is_dir() and (p.parent / 'configs').is_dir()]
+        assert len(candidates) == 1, 'Không xác định được thư mục LAnoBERT trong Dataset.'
+        source = candidates[0]
+    shutil.copytree(source, PROJECT, ignore=ignore)
+    print('Copied project from:', source)
+    print('Project files:', len(list(PROJECT.rglob('*'))))

In [ ]:
%pip install -q -r /kaggle/working/LAnoBERT/requirements.txt pytest

In [ ]:
import os
+    os.chdir(PROJECT)
+    print('Current directory:', Path.cwd())
+    print('Config exists:', Path(CONFIG).exists())
+    print('BGL directory:', Path('data/BGL').exists())
+    if Path('data/BGL').exists():
+        print(*sorted(str(p) for p in Path('data/BGL').iterdir()), sep='\n')

In [ ]:
raw_log = Path('data/BGL/BGL.log')
+    if not raw_log.exists() and DOWNLOAD_BGL:
+        subprocess.run(['bash', 'scripts/download_data.sh', 'bgl'], check=True)
+    assert raw_log.exists(), (
+        f'Thiếu {raw_log}. Upload BGL.log trong Dataset hoặc đặt DOWNLOAD_BGL=True '
+        'và bật Internet.'
+    )
+    print(raw_log, f'{raw_log.stat().st_size / (1024**3):.2f} GB')

## Kiểm tra code trước khi chạy dữ liệu lớn

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-v'], check=True)

In [ ]:
subprocess.run(['bash', '-n', 'scripts/ensure_data.sh', 'scripts/run_pipeline.sh'], check=True)
+    print('Shell syntax passed')

## Chuẩn bị dữ liệu BGL

Cell này có thể mất vài phút vì BGL có hàng triệu dòng. Có thể kiểm tra file đầu ra sau khi chạy.

In [ ]:
subprocess.run(['bash', 'scripts/ensure_data.sh', CONFIG], check=True)

In [ ]:
for path in [
+        'data/BGL/BGL_train_normal.raw',
+        'data/BGL/BGL_test.raw',
+        'data/BGL/BGL_test_label.log',
+        'data/BGL/BGL_train_normal_parsed.log',
+        'data/BGL/BGL_test_parsed.log',
+    ]:
+        file = Path(path)
+        print(f'{path}: {file.stat().st_size / (1024**2):.1f} MB' if file.exists() else f'MISSING: {path}')

## Train tokenizer

In [ ]:
subprocess.run([sys.executable, '-m', 'lanobert.tokenizer', '--config', CONFIG], check=True)

In [ ]:
tokenizer_dir = Path('outputs/BGL/tokenizer')
+    assert tokenizer_dir.exists(), f'Thiếu {tokenizer_dir}'
+    print(*sorted(str(p) for p in tokenizer_dir.iterdir()), sep='\n')

## Train LAnoBERT baseline

Cell này sử dụng cấu hình `EPOCHS`. Chạy thử với `EPOCHS=1` trước; sau đó đặt `EPOCHS=10` nếu cần thực nghiệm đầy đủ.

In [ ]:
import yaml
+    with open(CONFIG, encoding='utf-8') as handle:
+        runtime = yaml.safe_load(handle)
+    runtime.setdefault('train', {})['num_train_epochs'] = EPOCHS
+    runtime.setdefault('inference', {})['max_eval_samples'] = MAX_EVAL_SAMPLES
+    runtime_path = Path('configs/kaggle_runtime.yaml')
+    with runtime_path.open('w', encoding='utf-8') as handle:
+        yaml.safe_dump(runtime, handle, sort_keys=False)
+    print(runtime_path.read_text())

In [ ]:
subprocess.run([sys.executable, '-m', 'lanobert.train', '--config', 'configs/kaggle_runtime.yaml'], check=True)

In [ ]:
model_dir = Path('outputs/BGL/model')
+    print('Model exists:', model_dir.exists())
+    if model_dir.exists():
+        print(*sorted(str(p) for p in model_dir.rglob('*') if p.is_file())[-20:], sep='\n')

## Inference và đánh giá

In [ ]:
subprocess.run([sys.executable, '-m', 'lanobert.inference', '--config', 'configs/kaggle_runtime.yaml'], check=True)

In [ ]:
result_dir = Path('outputs/BGL/results')
+    assert result_dir.exists(), f'Thiếu {result_dir}'
+    print(*sorted(str(p) for p in result_dir.iterdir()), sep='\n')
+    reports = sorted(result_dir.glob('*report.txt'))
+    for report in reports:
+        print(f'\n===== {report} =====')
+        print(report.read_text()[:3000])

In [ ]:
import shutil
+    archive = shutil.make_archive('/kaggle/working/lanobert-results', 'zip', 'outputs')
+    print('Download artifact:', archive)